In [10]:
import numpy as np
import re
import polars as pl
import scipy
from operator import itemgetter

from langchain.callbacks.manager import CallbackManagerForRetrieverRun
from langchain.chains import LLMChain
from langchain.document_loaders import JSONLoader
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents.base import Document
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Set logging for the queries
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)


In [11]:
path = "/mnt/d/temp/user/ed/mart/mmplastic/20240725_154000_000000/data.json"

In [12]:
# question = "어떤 대학교가 미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구하는지 찾아줘"
question = "미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구와 관련된 미국 대학교를 찾고있어"
# question = "미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구를 한 대학 어디야?"

In [13]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            # [f"{d.metadata['seq_num']}-{d.metadata['sub_seq_num']} RANK:{i+1}\n{d.metadata['title'][:20]}:\n\n" + d.page_content for i, d in enumerate(docs)]
            [f"{d.metadata['seq_num']} RANK:{i+1}\n{d.metadata['title'][:64]}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["title"] = record.get("title")
    metadata["date"] = record.get("date")
    return metadata

loader = JSONLoader(
    file_path=path,
    jq_schema=".[]",
    content_key="content",
    text_content=True,
    metadata_func=metadata_func
)
text_splitter = RecursiveCharacterTextSplitter(
    separators="\n\n",
    chunk_size=200,
    chunk_overlap=0,
    keep_separator=True
)
embeddings = OllamaEmbeddings(
    model="tiger-gemma2"
)

In [14]:
llm = ChatOllama(
    model="tiger-gemma2",
    temperature=0.8,
    num_predict=320,
)

response = llm.invoke(question)
print(f"[{response.response_metadata['eval_duration'] / np.power(10., 9)} sec.]:\n" + "" + response.content)


[8.864785 sec.]:
미세플라스틱이 햇빛에 노출되면 오염줄기를 흡수하는 연구는 환경과 해양에서 미세플라스틱의 영향, 특히 식물 성장과 생태계 건강에 미치는 영향을 이해하는 데 중요한 역할을 합니다. 이러한 연구를 수행하는 미국 대학교를 찾고 있다면, 다음 학문 분야와 관련된 대학을 살펴보세요:

* **해양학 및 해양 과학:** 대부분의 해양대학은 오염줄기, 플라스틱 수리에 대한 장단점과 미세플라스틱이 해양 생태계에 미치는 영향 연구를 진행합니다. 예시로는 Scripps Institution of Oceanography (UCSD), Woods Hole Oceanographic Institution, University of Miami Rosenstiel School of Marine and Atmospheric Science 등을 들 수 있습니다.
* **지구과학 및 환경 과학:** 이 분야의 대학은 토양 오염, 공기 오염, 물 오염 등에 대한 연구를 진행하며 미세플라스틱이 지구 환경에 미치는 영향도 포함됩니다. 예시로는 Stanford University, Columbia University, Yale University 등을 들 수 있습니다.
* **농학과 균류학:** 이 분야의 대학은 식물 성장과 건강에 미세플라스틱이 미치는 영향, 특


In [15]:
from langchain_community.llms import Ollama
llm = Ollama(
    model="tiger-gemma2",
    temperature=0.8,
    num_predict=320,
)

from langchain.agents import AgentExecutor, create_tool_calling_agent, tool
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant"),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)
model = llm #ChatAnthropic(model="claude-3-opus-20240229")

@tool
def magic_function(input: int) -> int:
    """Applies a magic function to an input."""
    return input + 2

tools = [magic_function]

agent = create_tool_calling_agent(model, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({"input": "what is the value of magic_function(3)?"})

# Using with chat history
from langchain_core.messages import AIMessage, HumanMessage
agent_executor.invoke(
    {
        "input": "what's my name?",
        "chat_history": [
            HumanMessage(content="hi! my name is bob"),
            AIMessage(content="Hello Bob! How can I assist you today?"),
        ],
    }
)

ValueError: This function requires a .bind_tools method be implemented on the LLM.

In [ ]:
from llama_index.core.tools import FunctionTool


def get_weather(location: str) -> str:
    """Usfeful for getting the weather for a given location."""
    ...


tool = FunctionTool.from_defaults(
    get_weather,
    # async_fn=aget_weather,  # optional!
)

agent = ReActAgent.from_tools(tools, llm=llm, verbose=True)